In [3]:
!ls /kaggle/input/datasets/omswarooptm/dense-vit-model

dense_vit.pt


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torchvision.models import vit_b_16
from torch.utils.data import DataLoader
import copy
import random
import numpy as np

# =====================================================
# Device
# =====================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# =====================================================
# Seed
# =====================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

# =====================================================
# Data
# =====================================================
def get_cifar10(batch_size=32):

    transform_train = transforms.Compose([
        transforms.Resize(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.5,)*3, (0.5,)*3)
    ])

    transform_test = transforms.Compose([
        transforms.Resize(224),
        transforms.ToTensor(),
        transforms.Normalize((0.5,)*3, (0.5,)*3)
    ])

    trainset = torchvision.datasets.CIFAR10(
        root="./data",
        train=True,
        download=True,
        transform=transform_train
    )

    testset = torchvision.datasets.CIFAR10(
        root="./data",
        train=False,
        download=True,
        transform=transform_test
    )

    trainloader = DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2)
    testloader = DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=2)

    return trainloader, testloader

train_dl, test_dl = get_cifar10()

# =====================================================
# Evaluation
# =====================================================
@torch.no_grad()
def evaluate(model):
    model.eval()
    correct = 0
    total = 0

    for x, y in test_dl:
        x, y = x.to(device), y.to(device)
        out = model(x)
        pred = out.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.size(0)

    return correct / total

# =====================================================
# Sparse Wrapper
# =====================================================
class SparseViT(nn.Module):
    def __init__(self, base_model, keep_ratio=0.5):
        super().__init__()
        self.base = base_model
        self.keep_ratio = keep_ratio

    def forward(self, x):

        x = self.base._process_input(x)
        B, N, C = x.shape

        cls_token = self.base.class_token.expand(B, -1, -1)
        x = torch.cat((cls_token, x), dim=1)
        x = x + self.base.encoder.pos_embedding

        scores = x.norm(dim=-1)
        k = int(N * self.keep_ratio)
        topk = torch.topk(scores[:, 1:], k, dim=1).indices + 1

        cls_idx = torch.zeros(B, 1, dtype=torch.long, device=x.device)
        keep_idx = torch.cat([cls_idx, topk], dim=1)

        x = torch.gather(x, 1, keep_idx.unsqueeze(-1).expand(-1, -1, C))
        x = self.base.encoder.layers(x)
        x = self.base.encoder.ln(x)

        cls = x[:, 0]
        return self.base.heads(cls)

# =====================================================
# Load Teacher
# =====================================================
print("\nLoading Dense Teacher...")

teacher = vit_b_16(weights=None)
teacher.heads.head = nn.Linear(teacher.heads.head.in_features, 10)

teacher.load_state_dict(torch.load("/kaggle/input/datasets/omswarooptm/dense-vit-model/dense_vit.pt", map_location=device))
teacher.to(device)
teacher.eval()

teacher_acc = evaluate(teacher)
print("Teacher Accuracy:", teacher_acc)

# =====================================================
# Create Student
# =====================================================
student = SparseViT(copy.deepcopy(teacher), keep_ratio=0.5)

# =====================================================
# KD Training
# =====================================================
def train_kd(model, teacher, epochs=20, alpha=0.7, T=4.0):

    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5)
    scaler = torch.amp.GradScaler("cuda")

    best_acc = 0

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for x, y in train_dl:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()

            with torch.amp.autocast("cuda"):
                out = model(x)
                ce = F.cross_entropy(out, y)

                with torch.no_grad():
                    t_out = teacher(x)

                kd = F.kl_div(
                    F.log_softmax(out/T, dim=1),
                    F.softmax(t_out/T, dim=1),
                    reduction="batchmean"
                ) * (T*T)

                loss = alpha * kd + (1 - alpha) * ce

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item()

        val_acc = evaluate(model)
        print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(train_dl):.4f} - Val Acc: {val_acc:.4f}")

        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), "best_sparse_kd.pt")
            print("Saved best model.")

    return best_acc

# =====================================================
# Run KD
# =====================================================
print("\nTraining Sparse + KD...")
kd_acc = train_kd(student, teacher, epochs=20)

print("\nFinal Sparse + KD Accuracy:", kd_acc)

Using device: cuda


100%|██████████| 170M/170M [00:03<00:00, 49.2MB/s] 



Loading Dense Teacher...
Teacher Accuracy: 0.9689

Training Sparse + KD...
Epoch 1/20 - Loss: 0.7890 - Val Acc: 0.9248
Saved best model.
Epoch 2/20 - Loss: 0.4445 - Val Acc: 0.9236
Epoch 3/20 - Loss: 0.3381 - Val Acc: 0.9312
Saved best model.
Epoch 4/20 - Loss: 0.2745 - Val Acc: 0.9230
Epoch 5/20 - Loss: 0.2663 - Val Acc: 0.9332
Saved best model.
Epoch 6/20 - Loss: 0.2276 - Val Acc: 0.9263
Epoch 7/20 - Loss: 0.2106 - Val Acc: 0.9169
Epoch 8/20 - Loss: 0.2083 - Val Acc: 0.9275
Epoch 9/20 - Loss: 0.1989 - Val Acc: 0.9345
Saved best model.
Epoch 10/20 - Loss: 0.1836 - Val Acc: 0.9282
Epoch 11/20 - Loss: 0.1842 - Val Acc: 0.9253
Epoch 12/20 - Loss: 0.1849 - Val Acc: 0.9282
Epoch 13/20 - Loss: 0.1621 - Val Acc: 0.9207
Epoch 14/20 - Loss: 0.1714 - Val Acc: 0.9093
Epoch 15/20 - Loss: 0.1675 - Val Acc: 0.9224
Epoch 16/20 - Loss: 0.1557 - Val Acc: 0.9254
Epoch 17/20 - Loss: 0.1430 - Val Acc: 0.9288
Epoch 18/20 - Loss: 0.1538 - Val Acc: 0.9343
Epoch 19/20 - Loss: 0.1431 - Val Acc: 0.9324
Epoch 